In [3]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List

In [ ]:
@dataclass
class PetProfile:
    """
    Pet data
    """
    pet_id: str
    pet_name: Optional[str] = None
    breed: str                      # for searching any special attribute of the breed
    birth_date: datetime
    is_high_energy: bool            # e.g., Vizsla 
    postcode: str                   # For seasonal/geographic tick risk 
    streak: int                     # auto calculate number of checkin in (consecutive), reset if hasn't checkin more than 1 week (7 days)
    consistency: float               # Automatic check if has owner has checkin regularly
    worming_compliant: bool         # Non-compliance moves score to Action Needed 
    vaccination_current: bool       # Treatment currency feeds score 
    days_since_last_vet_visit: int  # Auto-calculated metric 
    
@dataclass
class VetInteractionLog:
    visited_vet: bool
    reason_for_visit: Optional[str] = None # Categorized reason informs score 

@dataclass
class DentalHealthLog:
    """Periodic check-in (Puppy teething, 6-month baseline, and 6-monthly ongoing)"""
    breath_odour: str             # e.g., "Noticeably worse", "Normal" 
    chewing_changes: str          # e.g., "Less interested", "Favouring one side" 
    gum_redness_bleeding: str     # e.g., "Noticed buildup/redness", "Normal" 
    professional_clean_due: bool  # True/False
    
@dataclass
class ToiletingLog:
    """Tracks bowel habits, frequency, and acute overrides"""
    frequency_changes: str    # e.g., "Normal", "More frequent", "Less frequent"
    consistency_changes: str  # e.g., "Normal", "Runny", "Hard" 
    indoor_accidents: bool    # True/False 
    blood_present: bool


@dataclass
class CheckInData:
    """The Weekly Check-in Data payload"""
    pet_id: str
    timestamp: datetime
    
    # Pre-Moment 1 Context
    vet_interaction: VetInteractionLog 
    
    # Core Weekly Vitals (Moment 1 & 2)
    energy: int                  # 1-10 scale (comparative to baseline) 
    appetite: int                # 1-10 scale 
    sleep_quality: int           # 1-10 scale 
    water_intake: str            # "Normal", "Increased", "Decreased" (Increased = High Risk) 
    activity_minutes: int        # Aggregated from Strava-style quick-logs 
    
    # Behaviour & Changes
    diet_changed: bool           # Elevated if coincides with health signal 
    behaviour_and_mood: str      # General mood observation 
    social_behaviour: str        # e.g., "Normal", "Withdrawn", "Clingy", "Aggressive" 
    
    # Physical & Specific Logs
    physical_health: Dict[str, int] # e.g., {"eyes": 10, "ears": 8, "coat": 9} 
    toileting: ToiletingLog         # Modular sub-class 
    
    # Parasite & Pain Checks (Moment 1 Additions)
    tick_check_completed: bool      # Passive score contributor if consistently done 
    skin_or_lump_issue_found: bool  # True/False 
    is_in_pain: bool                # Triggers quarterly or from combined flags 
    
    # Periodic & Monthly Inputs (Can be None/Null on standard weeks)
    weight: Optional[float] = None              # First check-in of each month [cite: 310]
    dental_log: Optional[DentalHealthLog] = None # Periodic (weeks 12-24, or 6-monthly) [cite: 213]
    
    # Closing Question (Breeder / Owner Support)
    owner_wellbeing_status: str     # Flags breeder dashboard if struggling 2+ weeks 


    

In [ ]:
class PetDiseaseIndicator:
    """
    Evaluates check-in data specifically for clinical red flags and maps them to potential disease categories for vet reference.
    """
    def __init__(self, pet: PetProfile):
        self.pet = pet

    def evaluate(self, current: CheckInData) -> List[Dict[str, str]]:
        clinical_flags = []

        # Increased Water Intake
        if current.water_intake == "Increased":
            clinical_flags.append({
                "trigger": "Increased Water Intake",
                "severity": "High",
                "action": "Requires bloodwork and urinalysis."
            })

        # Blood in Stool
        if current.toileting_blood:
            clinical_flags.append({
                "trigger": "Blood in Stool",
                "severity": "Acute",
                "action": "Immediate veterinary attention recommended."
            })

        # Confirmed Tick
        if current.tick_found:
            clinical_flags.append({
                "trigger": "Tick Found",
                "severity": "Acute",
                "action": "Immediate search for additional ticks and symptom monitoring; vet visit if symptomatic."
            })

        # Sudden Energy/Appetite Collapse 
        if current.energy <= 1 and current.appetite <= 1:
            clinical_flags.append({
                "trigger": "Lethargy & Anorexia Cluster",
                "severity": "Acute",
                "action": "Immediate veterinary attention recommended."
            })

        return clinical_flags

For now weight are indicated manually, weighting may changes base on pet age, breed,...

In [ ]:
class VitalityScoreEngine:
    """
    Vitality score calculator, using 4 week scores, need veterinians review for weight adjustment (this is just a demo)
    """
    def __init__(self, pet: PetProfile, history: list):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline (should use only 4 previous week
    def calculate_score(self, current: 'CheckInData') -> Dict[str, Any]:

        # "sudden collapse in appetite combined with lethargy" or blood or increased water
        is_lethargic_collapse = (current.energy <= 1 and current.appetite <= 1)
        if current.toileting_blood or current.water_intake == "Increased" or is_lethargic_collapse:
            return {"score": 40, "zone": "Action Needed", "override": True}
        
        # The first four weeks are a baseline establishment period.
        if len(self.history) < 4:
            return {"score": None, "status": "Building your baseline"}

        # Average pet score
        avg_energy_score = sum(h.energy for h in self.history) / 4
        avg_appetite_score = sum(h.appetite for h in self.history) / 4
        avg_sleep_score = sum(h.sleep_quality for h in self.history) / 4
        avg_activity_score = sum(h.activity_minutes for h in self.history) / 4

        # Calculate negative deviations (drops from normal)
        drop_energy = max(0, avg_energy_score - current.energy)
        drop_appetite = max(0, avg_appetite_score - current.appetite)
        drop_sleep = max(0, avg_sleep_score - current.sleep_quality)
        
        # Calculate activity drop as a percentage ratio
        drop_act_ratio = max(0, (avg_activity_score - current.activity_minutes) / max(1, avg_activity_score))

        # Weighting
        w_energy = 0.25
        w_appetite = 0.25
        w_sleep = 0.25
        w_activity = 0.25
        
        # High-energy breeds carry elevated activity weighting.
        if self.pet.is_high_energy:
            w_activity = 0.40 
            w_sleep = 0.10 # Adjust sleep down to balance the scale

        raw_score = 100.0

        # 5. DETECT SIGNALS VS NOISE (Elevated Combination Penalties)
        # Elevated when combined with other Moment 1 signals.
        has_appetite_signal = drop_appetite >= 1.0
        has_energy_signal = drop_energy >= 1.0
        has_sleep_signal = drop_sleep >= 1.0

        if has_appetite_signal and (has_energy_signal or has_sleep_signal):
            # This is a confirmed signal, apply a flat heavy deduction
            raw_score -= 25.0
        else:
            # This is standard variance/noise, apply normal weighted deductions
            raw_score -= (drop_appetite * 10 * w_appetite)
            raw_score -= (drop_energy * 10 * w_energy)
            raw_score -= (drop_sleep * 10 * w_sleep)

        # Always deduct for major activity drops based on breed weighting
        raw_score -= (drop_act_ratio * 30 * w_activity)

        # 6. FINALIZE SCORE
        final_score = max(0, min(100, int(raw_score)))

        return {
            "score": final_score, 
            "zone": self._get_zone(final_score),
            "status": "Active"
        }

    def _get_zone(self, score: int) -> str:
        """Returns the colour-coded zone to trigger Evie"""
        if score >= 85: return "Bright Green"
        if score >= 70: return "Medium Green"
        if score >= 50: return "Yellow/Watch Zone"
        return "Action Needed"
        